# Should we launch the new checkout?

**Product analytics / Experiment 03 / Clearly labeled simulation**

This notebook is an analytical walkthrough with executed outputs. It uses the same simulated user assignment, events, orders and refunds as the portfolio readout. The full pipeline is in `src/run_analysis.py`; run `python src/run_analysis.py --regenerate` to recreate raw data and all statistical results, then `python src/build_readout.py` for reports.

**Decision:** continue testing only if confirmation is feasible. Higher estimated conversion is insufficient when revenue and refund safety remain unresolved.


In [1]:
from pathlib import Path
import csv, json, sqlite3, sys
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'config.json').exists():
    raise RuntimeError('Run from the checkout-experiment project directory.')
sys.path.insert(0, str(ROOT / 'src'))
from statistics_tools import binary_effect, sample_ratio_test, decide
from run_analysis import load_database
from quality import validate_sources, reconcile
config = json.loads((ROOT / 'config.json').read_text())
metrics = json.loads((ROOT / 'results/metrics.json').read_text())
print('Source: fictional, locally generated randomized experiment')
print('Seed:', config['simulation']['seed'])
print('Purchase/refund windows:', config['measurement']['purchase_days'], '/', config['measurement']['refund_days'], 'days')


Source: fictional, locally generated randomized experiment
Seed: 314159
Purchase/refund windows: 7 / 30 days


## 1. Establish the unit of analysis and data quality

Randomization is independent 50:50 at user level. All assigned users belong in the denominator. Exact duplicate deliveries are removed; variant or time conflicts stop analysis. The following cells load the source CSVs, enforce the schema and execute the production SQL again in an in-memory database.


In [2]:
con = load_database(ROOT, config)
quality = validate_sources(con, config)
print(json.dumps(quality, indent=2))
print('Raw order/refund counts:', con.execute('SELECT COUNT(*) FROM orders').fetchone()[0], con.execute('SELECT COUNT(*) FROM refunds').fetchone()[0])


{
  "raw_assignment_rows": 24024,
  "assigned_users": 24000,
  "duplicate_extra_rows": 24,
  "conflicting_users": 0,
  "complete_followup": true
}
Raw order/refund counts: 3585 391


In [3]:
sql = (ROOT / 'sql/01_user_analysis.sql').read_text()
con.executescript(sql)
print(json.dumps(reconcile(con, config), indent=2))
arm_rows = con.execute('SELECT variant, COUNT(*), SUM(converted), AVG(gross_cents)/100.0, AVG(net_cents)/100.0 FROM user_analysis GROUP BY variant').fetchall()
print('\nVariant | users | buyers | gross RPU | after-refund RPU')
for variant,n,buyers,gross,net in arm_rows:
    print(f'{variant:8} | {n:5,d} | {buyers:5,d} | ${gross:.2f} | ${net:.2f}')


{
  "nonbuyers_retained": 20842,
  "excluded_orders": 179,
  "excluded_order_reasons": {
    "not_paid": 40,
    "pre_assignment": 25,
    "after_purchase_window": 114,
    "unassigned": 0
  },
  "reconciled_users": 24000,
  "reconciliation": "PASS"
}

Variant | users | buyers | gross RPU | after-refund RPU
control  | 11,911 | 1,532 | $14.87 | $13.58
new      | 12,089 | 1,626 | $14.81 | $13.30


### Why the SQL joins are ordered this way

Refund ledger → order totals → user totals → assigned users. This prevents split refunds from multiplying order value. The final joins keep users with no events or purchases; zero outcomes are explicit. Orders count in [assignment, assignment + 7 days), and refunds in [order, order + 30 days), with complete follow-up.


In [4]:
print(sql)


-- SQLite; UTC ISO timestamps normalized by the loader. All cutoffs are exclusive.
-- Execute the whole script: these CHECK constraints fail closed even in SQL-only use.
-- The configuration table must contain exactly one row (id=1).
DROP TABLE IF EXISTS temp.analysis_gate;
CREATE TEMP TABLE analysis_gate (
    check_name TEXT,
    violations INTEGER CHECK (violations = 0)
);
INSERT INTO analysis_gate VALUES
    ('single configuration row', ABS((SELECT COUNT(*) FROM experiment_config)-1));
INSERT INTO analysis_gate
SELECT 'assignment conflicts', COUNT(*) FROM (
    SELECT user_id FROM assignments GROUP BY user_id
    HAVING COUNT(DISTINCT variant)>1 OR COUNT(DISTINCT assigned_at)>1
);
INSERT INTO analysis_gate
SELECT 'missing user records', COUNT(*) FROM assignments a
LEFT JOIN users u ON a.user_id=u.user_id WHERE u.user_id IS NULL;
INSERT INTO analysis_gate
SELECT 'immature observation windows', COUNT(*) FROM assignments a
CROSS JOIN experiment_config c
WHERE strftime('%Y-%m-%dT%H:%M:

## 2. Check allocation before interpreting outcomes

The sample-ratio mismatch test uses unique assigned users. A p-value below 0.001 is an investigation flag. An unflagged result does not prove tracking or exposure delivery is correct.


In [5]:
n0, x0 = arm_rows[0][1:3]
n1, x1 = arm_rows[1][1:3]
allocation = sample_ratio_test(n0, n1, config['measurement']['expected_treatment_share'])
print(json.dumps(allocation, indent=2))


{
  "chi_square": 1.3201666666666667,
  "p_value": 0.2505621414242873,
  "expected_control": 12000.0,
  "expected_new": 12000.0,
  "observed_new_share": 0.5037083333333333
}


![Allocation counts](charts/allocation.svg)


## 3. Estimate conversion effects with uncertainty

Absolute effect: new conversion minus control conversion, reported in percentage points. Relative effect: new/control minus one. These are different quantities. The absolute interval is Newcombe's independent-proportions method using Wilson limits; the relative interval uses a log risk ratio.


In [6]:
conversion = binary_effect(x0, n0, x1, n1)
print(f"Control: {conversion['control']:.2%}; new: {conversion['new']:.2%}")
print(f"Absolute difference: {conversion['difference']*100:+.2f} pp")
print('95% absolute CI:', [round(x*100, 3) for x in conversion['ci95']], 'pp')
print(f"Relative lift: {conversion['relative_change']:+.2%}")
print('95% relative CI:', [round(x*100, 3) for x in conversion['relative_ci95']], '%')


Control: 12.86%; new: 13.45%
Absolute difference: +0.59 pp
95% absolute CI: [-0.267, 1.443] pp
Relative lift: +4.57%
95% relative CI: [-2.012, 11.6] %


![Conversion effect and uncertainty](charts/conversion.svg)

The interval includes a small decline and a commercially useful improvement. Inconclusive is not equivalent to no effect.


## 4. Revenue: retain zeroes and resample users

The pipeline performs 10,000 independent within-arm user resamples. Each selected user carries all of their orders and refunds. Gross and after-refund outcomes use the same user draws. This cell recomputes the reported intervals from the saved bootstrap draws, rather than copying numbers into the notebook.


In [7]:
draws = np.loadtxt(ROOT / 'results/bootstrap_draws.csv', delimiter=',', skiprows=1)
print('Bootstrap replicates:', len(draws))
for i,key in enumerate(['gross','net']):
    bounds = np.quantile(draws[:,i], [.025,.975])
    np.testing.assert_allclose(bounds, metrics['revenue'][key]['ci95'])
    print(f"{key}: difference ${metrics['revenue'][key]['difference']:+.2f}; 95% CI ${bounds[0]:+.2f} to ${bounds[1]:+.2f}")
print('All saved bootstrap intervals independently reconstructed.')


Bootstrap replicates: 10000
gross: difference $-0.06; 95% CI $-1.45 to $+1.27
net: difference $-0.28; 95% CI $-1.59 to $+0.98
All saved bootstrap intervals independently reconstructed.


![Revenue effects and uncertainty](charts/revenue.svg)


In [8]:
bridge = metrics['revenue_bridge']
for label,value in bridge['gross_contributions'].items():
    print(f'{label:24s} {value:+.4f} USD/user')
print(f"{'additional_refund_drag':24s} {bridge['additional_refund_drag']:+.4f} USD/user")
print(f"{'net_difference':24s} {bridge['net_difference']:+.4f} USD/user")


conversion               +0.6636 USD/user
orders_per_buyer         -0.1865 USD/user
average_order_value      -0.5345 USD/user
additional_refund_drag   -0.2242 USD/user
net_difference           -0.2815 USD/user


![Revenue decomposition](charts/revenue_drivers.svg)

This is an exact arithmetic decomposition averaged across six factor replacement orders. It is descriptive, not evidence that each factor independently changed because of the checkout.


## 5. Refund safety and the launch rule

The guardrail denominator is all assigned users. The buyer-only rate is descriptive because treatment can change who buys. Launch requires all three criteria. The margins are illustrative business assumptions, not historical preregistration or stakeholder approval.


In [9]:
print(json.dumps(metrics['refunds'], indent=2))
decision = decide(conversion['ci95'], metrics['revenue']['net']['ci95'], metrics['refunds']['ci95'],
                  config['decision']['net_revenue_loss_margin_usd'],
                  config['decision']['refund_rate_increase_margin'])
print(json.dumps(decision, indent=2))


{
  "control": 0.01242548904374108,
  "new": 0.014558689717925387,
  "difference": 0.0021332006741843074,
  "ci95": [
    -0.0007956078031935657,
    0.005073194183037699
  ],
  "relative_change": 0.1716794137176303,
  "relative_ci95": [
    -0.05699524895657926,
    0.45580671466466205
  ]
}
{
  "status": "CONTINUE TESTING \u2014 CONDITIONAL ON FEASIBILITY",
  "gates": {
    "conversion_superiority": false,
    "net_revenue_safety": false,
    "refund_safety": false
  }
}


![Refund uncertainty](charts/refunds.svg)


## 6. Is more testing worthwhile?

Conversion-only sample size is insufficient for the full launch decision. Below are planning scenarios using pilot variance, a one-sided 2.5% safety boundary and 80% power for each criterion. They do not imply 80% power to pass all criteria jointly.


In [10]:
print('Conversion users per arm:', metrics['planning']['approx_n_per_arm'])
print('\nRevenue safety if true revenue change is zero:')
for row in metrics['planning']['net_safety_scenarios']:
    if row['assumed_true_difference_usd'] == 0:
        print(f"Loss margin ${row['loss_margin_usd']:.2f}: {row['n_per_arm']:,} per arm; power at {metrics['planning']['reference_users_per_arm']:,}/arm = {row['power_at_reference_n']:.1%}")
print('\nRefund safety scenarios:')
for row in metrics['planning']['refund_safety_scenarios']:
    print(f"Assumed increase {row['assumed_true_difference']*100:.3f} pp: {row['n_per_arm']:,} per arm")


Conversion users per arm: 49838

Revenue safety if true revenue change is zero:
Loss margin $0.25: 644,548 per arm; power at 50,000/arm = 11.9%
Loss margin $0.50: 161,137 per arm; power at 50,000/arm = 34.5%
Loss margin $1.00: 40,285 per arm; power at 50,000/arm = 87.7%

Refund safety scenarios:
Assumed increase 0.000 pp: 21,404 per arm
Assumed increase 0.100 pp: 50,069 per arm
Assumed increase 0.213 pp: 278,063 per arm


![Next-test sample requirements](charts/sample_size.svg)


## 7. Exploratory checks without searching for a win

Shorter-window results use the same users and fixed raw data. They are exploratory point estimates only: no confidence intervals or hypothesis tests are calculated for them. They are not candidate windows to select after viewing outcomes and do not replace the configured primary analysis. Pre-assignment balance summaries do not establish randomization integrity by themselves.


In [11]:
for name in ['window_sensitivity','exploratory_balance']:
    print(name)
    print((ROOT / 'results' / f'{name}.csv').read_text())
con.close()


window_sensitivity
purchase_days,variant,assigned_users,buyers,conversion,gross_rpu,net_rpu,interpretation
1,control,11911,248,0.02082108974897154,2.3606556964150784,2.167352867097641,Exploratory point estimate only; no confidence interval or significance test
1,new,12089,230,0.01902556042683431,1.88385805277525,1.73021672594921,Exploratory point estimate only; no confidence interval or significance test
3,control,11911,697,0.058517336915456304,6.388387205104525,5.807862480060448,Exploratory point estimate only; no confidence interval or significance test
3,new,12089,739,0.06112995284969807,6.468416742493175,5.7564959880883455,Exploratory point estimate only; no confidence interval or significance test
7,control,11911,1532,0.12862060280413062,14.865366467970782,13.580378641591805,Exploratory point estimate only; no confidence interval or significance test
7,new,12089,1626,0.1345024402349243,14.808069319215814,13.298836959219125,Exploratory point estimate only; no confidence interval or

## Final recommendation and limitations

Retain the existing checkout. Consider a fresh independently planned test only after confirming economic tolerance, sample requirements and traffic capacity. If confirmation is too expensive, redesign the proposal. Do not keep adding users until an ordinary fixed-horizon interval becomes favorable.

This dataset is simulated, with stable identities, complete logging and no spillovers. Real evidence needs instrumentation audits. After-refund revenue excludes costs and long-term effects. Multiple subgroup searches can create false positives; no segment launch claims are made. Individual 95% intervals are not simultaneous 95% intervals.

See `docs/METHODOLOGY.md` for formulas, assumptions and references, and `docs/INTERVIEW_GUIDE.md` for a concise explanation of the project.
